## Calibration of document parsing pipeline for data extraction

In [1]:
import os
import os.path as osp
import sys

sys.path.append("../src")

In [2]:
from config import CFG
import pandas as pd
import pdfplumber

In [3]:
pdf_path = osp.join(CFG.DATA_DIR, os.listdir(CFG.DATA_DIR)[0])

pdf_path

'/Users/dric/projects/research/LLMs/chat_app/data/EDAN_2025_RESULTAT_NATIONAL_DETAILS.pdf'

In [22]:
def ingest_pdf(pdf_path):
    all_tables      = []
    all_tables_df   = []
    settings        = {
        "vertical_strategy": "lines",   # Excel usually exports with clear vertical lines
        "horizontal_strategy": "lines", # Use "text" if the rows don't have borders
        "snap_tolerance": 4,            # Merge lines that are nearly touching
        "join_tolerance": 4,            # Join broken lines
        "intersection_tolerance": 10,   # Helps with merged cells
    }
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables(table_settings=settings)
            for table in tables:
                all_tables.append(table)
                df = pd.DataFrame(table[2:], columns=table[:2])
                all_tables_df.append(df)
    
    try:
        # Merge and clean (Ensure numeric types for SQL aggregations)
        final_df    = pd.concat(all_tables_df)
        totals      = final_df.iloc[0]
        final_df    = final_df.drop(index=0).reset_index(drop=True)
        
        # Fix headers
        raw_headers = [h.lstrip().strip() if isinstance(h, str) else h for h in tables[0][:2]]
        new_headers = []

        for (r1, r2) in zip(raw_headers[0], raw_headers[1]):
            if r1 is None or r1=='':
                r1 = '-'
            if r2 is None or r2=='':
                r2 = '-'
            new_header = "".join([r1, r2])
            new_headers.append(new_header)
        
        final_df.columns = new_headers
    
        return totals, final_df, all_tables, all_tables_df
    
    except Exception as e:
        print(f" Error while merging data: {e}")
        return None, None, all_tables, all_tables_df
    

In [23]:
%%time
totals, df, tables, tables_df = ingest_pdf(pdf_path)

if df is not None:
    print(df.shape)

(1090, 16)
CPU times: user 5.28 s, sys: 59 ms, total: 5.34 s
Wall time: 5.34 s


In [24]:
df.head(10)

,REGION,-CIRCONSCRIPTION,--,-NB BV,-INSCRITS,-VOTANTS,TAUX DEPART.,BULL.NULS,SUF.EXPRIMES,BULL. BLANCSNOMBRE,-%,GROUPEMENTS / PARTISPOLITIQUES,-CANDIDATS / LISTES DE CANDIDATS,-SCORES,-%,--
0,A\nS\nS\nA\nIT\n-Y\nB\nE\nN\nG\nA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52 106,14070,"27,00%",388,13 682,76,"0,56%",INDEPENDANT,KOTO EHOU SOPIE,547,"4,00%",
1,None,None,None,None,None,None,None,None,None,None,None,RHDP,KOFFI AKA CHARLES,9 078,"66,35%",ELU(E)
2,None,None,None,None,None,None,None,None,None,None,None,INDEPENDANT,YEPIE KOUASSI THEODORE,58,"0,42%",
3,None,None,None,None,None,None,None,None,None,None,None,INDEPENDANT,TCHIMOU GNAMON BERTRAND,1 991,"14,55%",
4,None,None,None,None,None,None,None,None,None,None,None,INDEPENDANT,BOKA BOKA MAURICE,674,"4,93%",
5,None,None,None,None,None,None,None,None,None,None,None,ADCI,EDI DOFFOU PAUL,331,"2,42%",
6,None,None,None,None,None,None,None,None,None,None,None,FPI,N'GUESSAN KOTCHI REMI,474,"3,46%",
7,None,None,None,None,None,None,None,None,None,None,None,MGC,NANDO YAVO SERGE,453,"3,31%",
8,None,002,AGBOVILLE COMMUNE,133,48 710,12821,"26,32%",317,12 504,81,"0,65%",ADCI,OCHOU WROHOUM MARIE-PASCALE,296,"2,37%",
9,None,None,None,None,None,None,None,None,None,None,None,RHDP,DIMBA N'GOU PIERRE,10 675,"85,37%",ELU(E)


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1090 entries, 0 to 1089
Data columns (total 16 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   REGION                            24 non-null     object
 1   -CIRCONSCRIPTION                  183 non-null    object
 2   --                                183 non-null    object
 3   -NB BV                            183 non-null    object
 4   -INSCRITS                         170 non-null    object
 5   -VOTANTS                          170 non-null    object
 6   TAUX DEPART.                      170 non-null    object
 7   BULL.NULS                         170 non-null    object
 8   SUF.EXPRIMES                      170 non-null    object
 9   BULL. BLANCSNOMBRE                170 non-null    object
 10  -%                                170 non-null    object
 11  GROUPEMENTS / PARTISPOLITIQUES    1090 non-null   object
 12  -CANDIDATS / LISTES 

In [14]:
df["REGION"][0]

'A\nS\nS\nA\nIT\n-Y\nB\nE\nN\nG\nA'